# LA LA Stay · Airbnb Los Ángeles
### Análisis de datos — Andrés Arismendi (Uniandes)

**Fuente:** Inside Airbnb, `listings.csv.gz` de Los Ángeles (extracción del 14 al 16 de septiembre de 2026), guardado en `data/raw/`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)

In [ ]:
# Rutas del proyecto: funciona desde cualquier carpeta dentro del repositorio
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / 'data').is_dir():
            return folder
    return start

ROOT = find_project_root()
RAW_FILE = ROOT / 'data' / 'raw' / 'listings.csv.gz'
print('Archivo:', RAW_FILE)

### Cargar los datos
`id` y `host_id` se leen como **texto**: los IDs de Inside Airbnb tienen hasta 19 dígitos y, si se leen como número o pasan por Excel, pierden los últimos dígitos.

In [ ]:
df = pd.read_csv(RAW_FILE, compression='gzip', low_memory=False,
                 dtype={'id': str, 'host_id': str})

print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
df.head()

## 1 · Pasar las columnas JSON a columnas
Dos columnas guardan texto en formato JSON: `price_quote_raw` (objeto) y `amenities` (lista). Primero lo confirmamos.

In [ ]:
import json

def looks_like_json(series, sample=500):
    values = series.dropna().astype(str).head(sample)
    if values.empty or not values.str.strip().str[:1].isin(['{', '[']).all():
        return False
    try:
        values.map(json.loads)
        return True
    except (ValueError, TypeError):
        return False

json_columns = [col for col in df.select_dtypes('object') if looks_like_json(df[col])]
print('Columnas JSON:', json_columns)
print('Shape inicial:', df.shape)

### 1.1 · `price_quote_raw`: cada llave es una columna
Cada fila trae `{"quote": {...}}` con 17 llaves. Cada llave pasa a una columna con el prefijo `quote_`.

La llave `raw_price_line_items` es a su vez una **lista** de cobros (hasta 12 por anuncio), y cada cobro tiene 4 llaves: `amount`, `item_type`, `description` y `price_string`. Se expande por posición: `quote_line_item_1_amount`, `quote_line_item_1_item_type`, …, `quote_line_item_12_price_string`. Si un anuncio tiene menos cobros, las columnas sobrantes quedan vacías.

In [ ]:
quotes = df['price_quote_raw'].map(lambda raw: json.loads(raw)['quote'] if pd.notna(raw) else {})
quote_df = pd.json_normalize(quotes.tolist()).add_prefix('quote_').set_index(df.index)

# Expandir la lista de cobros por posición
line_items = quote_df.pop('quote_raw_price_line_items')
item_keys = ['amount', 'item_type', 'description', 'price_string']
max_items = int(line_items.map(lambda x: len(x) if isinstance(x, list) else 0).max())

items_df = pd.DataFrame(
    [[(items[i].get(key) if isinstance(items, list) and i < len(items) else None)
      for i in range(max_items) for key in item_keys]
     for items in line_items],
    columns=[f'quote_line_item_{i + 1}_{key}' for i in range(max_items) for key in item_keys],
    index=df.index,
)

print('Llaves de quote:', quote_df.shape[1] + 1, '(incluye raw_price_line_items)')
print('Máximo de cobros por anuncio:', max_items, '->', items_df.shape[1], 'columnas de cobros')

In [ ]:
# Tipos de dato: montos a número, fechas a fecha, banderas a booleano
money = ['quote_taxes', 'quote_total_price', 'quote_discount_amount', 'quote_price_per_night',
         'quote_nightly_subtotal', 'quote_discounted_subtotal']
quote_df[money] = quote_df[money].apply(pd.to_numeric, errors='coerce')
date_cols = [c for c in quote_df.columns if c.endswith('_date')]
quote_df[date_cols] = quote_df[date_cols].apply(pd.to_datetime)
for col in ['quote_is_available', 'quote_is_monthly_priced']:
    quote_df[col] = quote_df[col].astype('boolean')

amount_cols = [c for c in items_df.columns if c.endswith('_amount')]
items_df[amount_cols] = items_df[amount_cols].apply(pd.to_numeric, errors='coerce')

df = pd.concat([df.drop(columns=['price_quote_raw']), quote_df, items_df], axis=1)
print('Shape después de price_quote_raw:', df.shape)

In [ ]:
# Llaves que vienen vacías en TODAS las filas (se conservan, pero conviene saberlo)
empty_quote = [c for c in quote_df.columns if quote_df[c].isna().all()]
print('Llaves de quote siempre vacías:', empty_quote)

### 1.2 · `amenities`
`amenities` es una **lista de textos, sin llaves** (por ejemplo `["Wifi", "Kitchen", "Pool"]`). Pasarla a columnas significa crear **una columna por servicio** (1 si el anuncio lo tiene, 0 si no). Aquí solo se calcula cuántas columnas resultarían, sin crearlas todavía.

In [ ]:
amenities_list = df['amenities'].map(lambda raw: [a.strip().lower() for a in json.loads(raw)] if pd.notna(raw) else [])
n_amenities = amenities_list.explode().nunique()
print(f'Servicios distintos: {n_amenities:,}')
print(f'Shape si cada servicio fuera una columna: ({df.shape[0]:,}, {df.shape[1] - 1 + n_amenities:,})')

### 1.3 · ¿Qué valores toma `amenities`?
Tabla de frecuencia de **todos** los servicios: en cuántos anuncios aparece cada uno y qué porcentaje del total representa.

In [ ]:
amenity_freq = (amenities_list.explode().dropna()
                .value_counts()
                .rename_axis('amenity')
                .reset_index(name='anuncios'))
amenity_freq['% anuncios'] = (amenity_freq['anuncios'] / len(df) * 100).round(2)

print(f'Servicios distintos: {len(amenity_freq):,}')
print(f'Anuncios sin ningún servicio: {(amenities_list.str.len() == 0).sum()}')
amenity_freq.head(60)

**Distribución por frecuencia:** cuántos servicios hay en cada rango de popularidad.

In [ ]:
n = len(df)
bands = pd.cut(amenity_freq['anuncios'],
               bins=[0, 1, 0.001 * n, 0.01 * n, 0.05 * n, 0.90 * n, n],
               labels=['1 solo anuncio', '2 anuncios a 0,1 %', '0,1 % – 1 %',
                       '1 % – 5 %', '5 % – 90 %', 'más de 90 %'])
band_table = bands.value_counts().sort_index().to_frame('servicios')
band_table['% de servicios'] = (band_table['servicios'] / len(amenity_freq) * 100).round(1)
band_table

**Variantes del mismo servicio:** muchos servicios son el mismo con distinta marca, tamaño o detalle. Ejemplos por palabra clave:

In [ ]:
for keyword in ['tv', 'wifi', 'shampoo', 'pool', 'parking', 'coffee']:
    variants = amenity_freq[amenity_freq['amenity'].str.contains(keyword, regex=False)]
    print(f"'{keyword}': {len(variants):,} variantes -> {variants['amenity'].head(6).tolist()}")

print('\nEjemplos de servicios que aparecen en un solo anuncio:')
print(amenity_freq.loc[amenity_freq['anuncios'] == 1, 'amenity'].sample(8, random_state=1).tolist())

### 1.4 · Unificar las variantes de `amenities`
Cada servicio unificado se define con una **expresión regular** sobre el texto en minúsculas. Un anuncio tiene el servicio si **alguno** de sus textos coincide. Ejemplos:
- `tv` agrupa "tv", "hdtv", "55 inch hdtv with netflix"…
- `pool` agrupa "pool", "private pool", "shared outdoor pool…", pero **no** "pool table" ni "pool view".
- `washer` excluye "dishwasher"; `dryer` excluye "hair dryer".

In [ ]:
import re

AMENITY_RULES = {
    # Básicos
    'wifi':              r'\bwi-?fi\b',
    'kitchen':           r'\bkitchen\b',
    'tv':                r'\b(?:hd)?tv\b',
    'workspace':         r'workspace',
    'coffee_maker':      r'coffee maker|espresso|nespresso|keurig',
    # Clima
    'air_conditioning':  r'air conditioning|\bac\b|a/c',
    'heating':           r'heating|\bheater\b',
    'fireplace':         r'fireplace',
    'ceiling_fan':       r'ceiling fan',
    # Lavado y baño
    'washer':            r'(?<!dish)washer',
    'dryer':             r'(?<!hair )\bdryer\b',
    'dishwasher':        r'dishwasher',
    'bathtub':           r'bathtub|soaking tub',
    # Exterior
    'pool':              r'\bpool\b(?! table)(?! view)',
    'hot_tub':           r'hot tub|jacuzzi',
    'patio_balcony':     r'patio|balcony',
    'backyard':          r'backyard',
    'bbq_grill':         r'\bbbq\b|barbecue|\bgrill\b',
    'outdoor_dining':    r'outdoor dining|outdoor furniture',
    'fire_pit':          r'fire pit',
    'garden':            r'\bgarden\b',
    # Acceso y estacionamiento
    'free_parking':      r'free .*parking',
    'paid_parking':      r'paid .*parking',
    'ev_charger':        r'ev charger',
    'private_entrance':  r'private entrance',
    'self_check_in':     r'self check-?in|keypad|lockbox|smart lock',
    'elevator':          r'\belevator\b',
    # Ocio
    'gym':               r'\bgym\b|exercise equipment',
    'sound_system':      r'sound system|bluetooth speaker',
    'game_console':      r'game console|playstation|\bps[45]\b|xbox|nintendo',
    # Ubicación / vistas
    'beach_access':      r'beach access|beachfront',
    'ocean_view':        r'ocean view|sea view|beach view',
    'mountain_view':     r'mountain view',
    'city_view':         r'city skyline view|city view',
    # Familias y políticas
    'crib':              r'\bcrib\b|pack .n play',
    'pets_allowed':      r'pets allowed',
    'long_term_stays':   r'long term stays allowed',
}

joined = amenities_list.map(' | '.join)          # todos los servicios del anuncio en un texto
for name, pattern in AMENITY_RULES.items():
    df[f'am_{name}'] = joined.str.contains(pattern, regex=True).astype(int)
df['amenities_count'] = amenities_list.str.len()

am_cols = [f'am_{name}' for name in AMENITY_RULES]
print(f'{len(am_cols)} servicios unificados creados | Shape: {df.shape}')

In [ ]:
# Verificación: qué textos originales quedaron dentro de algunos servicios unificados
exploded = amenity_freq.copy()
for name in ['tv', 'pool', 'washer', 'dryer', 'free_parking']:
    hits = exploded[exploded['amenity'].str.contains(AMENITY_RULES[name], regex=True)]
    print(f"{name}: {len(hits):,} variantes, p. ej. {hits['amenity'].head(4).tolist()}")

### 1.5 · Frecuencia y relación con el precio
Para comparar precios usamos el **mercado objetivo del Charter**: hosts con 1–2 anuncios, con precio y **sin precio mensual** (`quote_is_monthly_priced = False`).
Se compara el **precio mediano** de los anuncios con y sin cada servicio. Es una comparación **descriptiva, no causal**: un servicio puede ir junto con propiedades más grandes o mejor ubicadas.

In [ ]:
df['price_num'] = pd.to_numeric(df['price'].str.replace(r'[\$,]', '', regex=True), errors='coerce')
target = df[df['calculated_host_listings_count'].between(1, 2)
            & df['price_num'].notna()
            & (df['quote_is_monthly_priced'] == False)]
print(f'Anuncios del mercado objetivo: {len(target):,}')

rows = []
for col in am_cols:
    has = target[col] == 1
    rows.append({
        'servicio': col.removeprefix('am_'),
        '% anuncios (todos)': df[col].mean() * 100,
        '% mercado objetivo': has.mean() * 100,
        'mediana con': target.loc[has, 'price_num'].median(),
        'mediana sin': target.loc[~has, 'price_num'].median(),
    })
amenity_summary = pd.DataFrame(rows)
amenity_summary['diferencia %'] = (amenity_summary['mediana con'] / amenity_summary['mediana sin'] - 1) * 100
amenity_summary['entre 5 % y 90 %'] = amenity_summary['% mercado objetivo'].between(5, 90)
amenity_summary = amenity_summary.sort_values('% mercado objetivo', ascending=False).round(1).reset_index(drop=True)
amenity_summary

### 1.6 · Servicios seleccionados
Se descartan 9 servicios unificados:
- **Casi universales (más del 90 % del mercado objetivo):** `wifi`, `tv`, `free_parking`. No diferencian un anuncio de otro.
- **Muy raros (menos del 5 %):** `ocean_view`, `game_console`.
- **Diferencia de precio pequeña o ambigua:** `ceiling_fan`, `elevator`, `private_entrance` (5 %–8 %) y `paid_parking` (negativa: suele indicar zonas densas, no un servicio que agregue valor).

Se conservan los demás servicios más `amenities_count`, y se elimina la columna JSON original `amenities`.

In [ ]:
DROP_AMENITIES = ['wifi', 'tv', 'free_parking', 'ocean_view', 'game_console',
                  'ceiling_fan', 'elevator', 'private_entrance', 'paid_parking']

df = df.drop(columns=[f'am_{name}' for name in DROP_AMENITIES] + ['amenities'])
am_cols = [col for col in df.columns if col.startswith('am_')]

print(f'Servicios conservados ({len(am_cols)}):')
print([col.removeprefix('am_') for col in am_cols])
print(f'\nShape: {df.shape}')

## 2 · Selección de columnas
### 2.1 · Variables derivadas antes de eliminar
Antes de quitar las columnas de la cotización se calculan dos variables útiles:
- `quote_nights`: noches que se cotizaron (de 1 a 365, según las noches mínimas).
- `base_nightly_rate`: **tarifa publicada por noche sin descuentos** = `quote_nightly_subtotal / quote_nights`. Existe solo para los anuncios con precio diario (no mensuales). Es una posible variable objetivo alternativa a `price`, que ya trae descuentos aplicados.

In [ ]:
df['quote_nights'] = (df['quote_requested_checkout_date'] - df['quote_requested_checkin_date']).dt.days
# Solo tiene sentido con precio diario: en las cotizaciones mensuales no hay tarifa por noche
df['base_nightly_rate'] = (df['quote_nightly_subtotal'] / df['quote_nights']).where(df['quote_is_monthly_priced'] == False)

print(f"Anuncios con tarifa base: {df['base_nightly_rate'].notna().sum():,}")
print(pd.crosstab(df['base_nightly_rate'].notna(), df['quote_is_monthly_priced'],
                  rownames=['tiene tarifa base'], colnames=['precio mensual']))

### 2.2 · Columnas a eliminar, por grupo
Cada grupo indica **por qué** se elimina. Al final se verifica que ninguna columna quede sin clasificar.

In [ ]:
always_empty = [col for col in df.columns if df[col].isna().all()]

DROP_GROUPS = {
    'Vacías en todas las filas': always_empty,
    'Metadatos de la extracción': [
        'scrape_id', 'last_scraped', 'calendar_last_scraped', 'source', 'quote_currency',
        'quote_requested_checkin_date', 'quote_requested_checkout_date'],
    'URLs, identificadores secundarios y texto libre': [
        'listing_url', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url',
        'host_picture_url', 'host_name', 'name', 'description', 'host_about', 'host_location', 'license'],
    'Derivadas del precio (fuga hacia la variable objetivo)': [
        'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price',
        'price_quote_price_per_night', 'estimated_revenue_l365d', 'quote_total_price',
        'quote_price_per_night', 'quote_taxes', 'quote_discount_amount', 'quote_nightly_subtotal',
        'quote_discounted_subtotal'] + [c for c in df.columns if c.startswith('quote_line_item_')],
    'Redundantes': [
        'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights',
        'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',
        'host_listings_count', 'calculated_host_listings_count_entire_homes',
        'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms',
        'hosts_time_as_user_years', 'hosts_time_as_user_months'],
    'Desempeño del anuncio (un anfitrión nuevo no lo conoce)': [
        'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
        'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
        'review_scores_value', 'number_of_reviews_l30d', 'number_of_reviews_ly', 'reviews_per_month',
        'first_review', 'availability_30', 'availability_60', 'availability_90', 'availability_eoy',
        'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
        'hosts_time_as_host_years', 'hosts_time_as_host_months', 'quote_is_available'],
}

# Una columna puede estar en "vacías" y en otro grupo; se cuenta una sola vez
to_drop = list(dict.fromkeys(col for cols in DROP_GROUPS.values() for col in cols))
missing = [col for col in to_drop if col not in df.columns]
assert not missing, f'Columnas inexistentes: {missing}'

summary = pd.DataFrame({'grupo': list(DROP_GROUPS),
                        'columnas': [len(cols) for cols in DROP_GROUPS.values()]})
display(summary)
print(f'Total a eliminar (sin repetir): {len(to_drop)}')

In [ ]:
df = df.drop(columns=to_drop)
print(f'Shape: {df.shape}')

### 2.3 · Columnas conservadas

In [ ]:
KEEP_GROUPS = {
    'Clave': ['id'],
    'Variables objetivo (candidatas)': ['price_num', 'base_nightly_rate'],
    'Características de la propiedad': [
        'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude',
        'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms_text'],
    'Servicios': [c for c in df.columns if c.startswith('am_')] + ['amenities_count'],
    'Filtros del mercado objetivo': [
        'calculated_host_listings_count', 'minimum_nights', 'maximum_nights',
        'quote_is_monthly_priced', 'quote_nights'],
    'Actividad (solo análisis, no van al modelo)': [
        'number_of_reviews', 'number_of_reviews_ltm', 'last_review', 'availability_365',
        'estimated_occupancy_l365d', 'has_availability'],
}
classified = [col for cols in KEEP_GROUPS.values() for col in cols]
unclassified = sorted(set(df.columns) - set(classified))
assert not unclassified, f'Columnas sin clasificar: {unclassified}'

df = df[classified]
df['last_review'] = pd.to_datetime(df['last_review'])
df['has_availability'] = df['has_availability'].map({'t': True, 'f': False}).astype('boolean')

display(pd.DataFrame({'grupo': list(KEEP_GROUPS), 'columnas': [len(c) for c in KEEP_GROUPS.values()]}))
print(f'Shape final: {df.shape}')

## 3 · Limpieza
Igual que el equipo del TEC, **no se borran filas**: los casos problemáticos se **marcan** con columnas 0/1 y se excluyen solo al entrenar.

Los pasos 3.2 y 3.4 reutilizan la lógica del notebook de limpieza del TEC (`Data_LA_Clean.ipynb`, Lorena Martínez). Para que los resultados sean comparables con los suyos, las medianas y los conteos de esos pasos se calculan sobre la misma población: **hosts con 1–2 anuncios**.

### 3.1 · Marcas del mercado objetivo
| Marca | Definición |
|---|---|
| `is_small_host` | El anfitrión tiene 1 o 2 anuncios (usuario del Project Charter). |
| `has_price` | El anuncio tiene precio. |
| `is_monthly` | La cotización tiene precio mensual (`quote_is_monthly_priced`). |
| `in_target_market` | `is_small_host` **y** `has_price` **y no** `is_monthly`. |
| `is_hotel` | `room_type` es *Hotel room*: no corresponde a un anfitrión independiente. |
| `is_inactive` | Sin reseñas en los últimos 12 meses **y** ocupación estimada de 0 días. |
| `price_outlier` | Precio menor a USD 20 o mayor al percentil 99 **del mercado objetivo**. |

In [ ]:
df['is_small_host'] = df['calculated_host_listings_count'].between(1, 2).astype(int)
df['has_price'] = df['price_num'].notna().astype(int)
df['is_monthly'] = df['quote_is_monthly_priced'].fillna(False).astype(int)
df['in_target_market'] = ((df['is_small_host'] == 1) & (df['has_price'] == 1)
                          & (df['is_monthly'] == 0)).astype(int)
df['is_hotel'] = (df['room_type'] == 'Hotel room').astype(int)
df['is_inactive'] = ((df['number_of_reviews_ltm'] == 0)
                     & (df['estimated_occupancy_l365d'] == 0)).astype(int)

PRICE_MIN = 20
PRICE_MAX = df.loc[df['in_target_market'] == 1, 'price_num'].quantile(0.99)
df['price_outlier'] = (df['has_price'].astype(bool)
                       & ((df['price_num'] < PRICE_MIN) | (df['price_num'] > PRICE_MAX))).astype(int)
print(f'Rango de precio aceptado: USD {PRICE_MIN} – {PRICE_MAX:,.2f} (percentil 99 del mercado objetivo)')

flags = ['is_small_host', 'has_price', 'is_monthly', 'in_target_market', 'is_hotel', 'is_inactive', 'price_outlier']
target = df['in_target_market'] == 1
pd.DataFrame({'todos los anuncios': df[flags].sum(),
              'dentro del mercado objetivo': df.loc[target, flags].sum()})

### 3.2 · Habitaciones y camas vacías
Lógica del TEC: se rellenan con la **mediana de anuncios similares** (mismo `room_type` y misma capacidad `accommodates`). Si ese grupo tiene menos de 10 anuncios, se usa la mediana del `room_type`. Se guarda el valor original y una marca de imputación. Se aplica igual a `beds`, que el TEC no usaba.

In [ ]:
small = df['is_small_host'] == 1

def impute_by_similar(column, min_group=10):
    group = ['room_type', 'accommodates']
    stats = df[small].groupby(group)[column].agg(['median', 'count']).reset_index()
    room_median = df[small].groupby('room_type')[column].median()
    stats.loc[stats['count'] < min_group, 'median'] = stats['room_type'].map(room_median)

    fill = df[group].merge(stats[group + ['median']], on=group, how='left')['median'].to_numpy()
    fill = np.where(np.isnan(fill), df['room_type'].map(room_median), fill)   # combinaciones no vistas

    df[f'{column}_original'] = df[column]
    df[f'{column}_imputed'] = df[column].isna().astype(int)
    df[column] = df[column].fillna(pd.Series(fill, index=df.index))

for column in ['bedrooms', 'beds']:
    impute_by_similar(column)
    print(f"{column}: {df[f'{column}_imputed'].sum():,} valores rellenados "
          f"({df.loc[small, f'{column}_imputed'].sum():,} en hosts 1–2)")

### 3.3 · Baños: de texto a número
`bathrooms_text` mezcla número y tipo (*"1.5 shared baths"*, *"Private half-bath"*). Se separa en:
- `bathrooms`: número de baños; *half-bath* = 0,5.
- `bathroom_shared`: 1 si el baño es compartido.

Los 74 vacíos se rellenan con la misma lógica de 3.2.

In [ ]:
text = df['bathrooms_text'].str.lower()
df['bathrooms'] = pd.to_numeric(text.str.extract(r'^(\d+(?:\.\d+)?)')[0], errors='coerce')
df.loc[text.str.contains('half-bath', na=False), 'bathrooms'] = 0.5
df['bathroom_shared'] = text.str.contains('shared', na=False).astype(int)

impute_by_similar('bathrooms')
df = df.drop(columns=['bathrooms_original'])           # el original queda en bathrooms_text

print(f"Baños rellenados: {df['bathrooms_imputed'].sum()}")
print(df[['bathrooms_text', 'bathrooms', 'bathroom_shared']].drop_duplicates()
        .sort_values('bathrooms').head(12).to_string(index=False))

### 3.4 · Tipos de propiedad raros y vecindarios pequeños
**Tipos de propiedad** (lógica del TEC): los tipos con menos de 100 anuncios se agrupan según `room_type`. Corrección respecto al TEC: allí los hoteles raros quedaban dentro de *Shared room*; aquí van en su propio grupo, *Unique hotel room*.

**Vecindarios** (lógica del TEC, calculada con código): los vecindarios con menos de 10 anuncios se unen al **vecindario más cercano** con 10 o más, medido como distancia entre centros (mediana de latitud y longitud de sus anuncios). El TEC escribió el mapa de uniones a mano; aquí se calcula, para que sea reproducible.

In [ ]:
# Tipos de propiedad
type_counts = df.loc[small, 'property_type'].value_counts()
rare = df['property_type'].map(type_counts).fillna(0) < 100
rare_label = df['room_type'].map({'Entire home/apt': 'Unique entire place',
                                  'Private room': 'Unique private room',
                                  'Shared room': 'Shared room',
                                  'Hotel room': 'Unique hotel room'})
df['property_type_model'] = np.where(rare, rare_label, df['property_type'])
print(f"Tipos de propiedad (hosts 1–2): {df.loc[small, 'property_type'].nunique()} -> "
      f"{df.loc[small, 'property_type_model'].nunique()}")

In [ ]:
# Vecindarios
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

hood_counts = df.loc[small, 'neighbourhood_cleansed'].value_counts()
centers = df[small].groupby('neighbourhood_cleansed')[['latitude', 'longitude']].median()
large = hood_counts[hood_counts >= 10].index

neighbourhood_map = {}
for hood in df['neighbourhood_cleansed'].unique():
    if hood_counts.get(hood, 0) >= 10:
        continue
    lat, lon = (centers.loc[hood] if hood in centers.index
                else df.loc[df['neighbourhood_cleansed'] == hood, ['latitude', 'longitude']].median())
    distances = haversine_km(lat, lon, centers.loc[large, 'latitude'], centers.loc[large, 'longitude'])
    neighbourhood_map[hood] = distances.idxmin()

df['neighbourhood_model'] = df['neighbourhood_cleansed'].replace(neighbourhood_map)
print(f"Vecindarios (hosts 1–2): {df.loc[small, 'neighbourhood_cleansed'].nunique()} -> "
      f"{df.loc[small, 'neighbourhood_model'].nunique()}")
print(f'Vecindarios unidos a otro: {len(neighbourhood_map)}')
pd.Series(neighbourhood_map, name='unido a').rename_axis('vecindario').head(10)

### 3.5 · Validaciones automáticas
Si alguna condición falla, el notebook se detiene con un mensaje. Las inconsistencias que no se pueden corregir con certeza se **marcan** en vez de detener el proceso.

In [ ]:
# Estructura
assert len(df) == 43_781, 'Cambió el número de filas'
assert df['id'].is_unique, 'Hay IDs repetidos'
assert df['id'].str.fullmatch(r'\d+').all(), 'Hay IDs que no son solo dígitos'
assert df['is_small_host'].sum() == 20_541, 'El filtro de hosts no coincide con el del TEC (20.541)'

# Rangos válidos
assert df['latitude'].between(33.2, 34.9).all() and df['longitude'].between(-119.0, -117.6).all(), \
    'Hay coordenadas fuera del condado de Los Ángeles'
assert (df['accommodates'] >= 1).all(), 'Hay anuncios con capacidad menor a 1'
assert (df.loc[df['has_price'] == 1, 'price_num'] > 0).all(), 'Hay precios no positivos'
assert (df['bathrooms'] >= 0).all() and (df['bedrooms'] >= 0).all(), 'Hay baños o habitaciones negativos'

# Sin vacíos en las variables del modelo
model_cols = ['bedrooms', 'beds', 'bathrooms', 'accommodates', 'neighbourhood_model',
              'property_type_model', 'room_type', 'amenities_count']
nulls = df[model_cols].isna().sum()
assert nulls.sum() == 0, f'Quedan vacíos en variables del modelo: {nulls[nulls > 0].to_dict()}'

# Inconsistencia que se marca: más habitaciones que huéspedes (no se sabe cuál de los dos valores está mal)
df['bedrooms_gt_guests'] = (df['bedrooms'] > df['accommodates']).astype(int)
print('Todas las validaciones pasaron.')
print(f"Anuncios con más habitaciones que huéspedes: {df['bedrooms_gt_guests'].sum()} "
      f"({df.loc[small, 'bedrooms_gt_guests'].sum()} en hosts 1–2)")
print(f'Shape: {df.shape}')

### 3.6 · DataFrame limpio
Hasta aquí la limpieza solo **marcó** problemas. Ahora se crea **`df_limpio`**, quitando las filas que la limpieza descarta:

| Se queda si… | Marca |
|---|---|
| El anfitrión tiene 1 o 2 anuncios (usuario del Project Charter) | `is_small_host = 1` |
| Tiene precio | `has_price = 1` |
| No es un hotel | `is_hotel = 0` |
| No tiene precio extremo | `price_outlier = 0` |

La **actividad** y el **tipo de precio** no se filtran aquí: son los dos criterios con los que se subdivide `df_limpio` en el paso 3.7. `df` completo se conserva sin cambios.

In [ ]:
steps = [
    ('Todos los anuncios', pd.Series(True, index=df.index)),
    ('Hosts con 1–2 anuncios', df['is_small_host'] == 1),
    ('Con precio', (df['is_small_host'] == 1) & (df['has_price'] == 1)),
    ('Sin hoteles', (df['is_small_host'] == 1) & (df['has_price'] == 1) & (df['is_hotel'] == 0)),
    ('Sin precios extremos', (df['is_small_host'] == 1) & (df['has_price'] == 1) & (df['is_hotel'] == 0)
                             & (df['price_outlier'] == 0)),
]
funnel = pd.DataFrame([(name, int(mask.sum())) for name, mask in steps], columns=['paso', 'anuncios'])
funnel['excluidos en el paso'] = (-funnel['anuncios'].diff()).fillna(0).astype(int)

df_limpio = df[steps[-1][1]].copy()
print(f'df_limpio: {df_limpio.shape}')
funnel

### 3.7 · Dividir `df_limpio` en diario y mensual
`df_limpio` se divide en **dos DataFrames** según el tipo de precio, porque son mercados distintos (Airbnb cobra precio mensual cuando la cotización es de 28 noches o más):
- **`df_diario`**: anuncios con precio por noche.
- **`df_mensual`**: anuncios con precio mensual.

Dentro de cada uno, la variable **`activo`** indica si el anuncio tuvo actividad en el último año (1) o no (0: sin reseñas en 12 meses y con 0 días de ocupación).

La población de entrenamiento del modelo del tablero es **`df_diario` con `activo = 1`**.

In [ ]:
df_limpio['activo'] = 1 - df_limpio['is_inactive']
df['activo'] = 1 - df['is_inactive']

df_diario = df_limpio[df_limpio['is_monthly'] == 0].copy()
df_mensual = df_limpio[df_limpio['is_monthly'] == 1].copy()
assert len(df_diario) + len(df_mensual) == len(df_limpio), 'La división no cubre df_limpio'

# Marca de entrenamiento: anuncios diarios activos
df['in_training'] = df['id'].isin(df_diario.loc[df_diario['activo'] == 1, 'id']).astype(int)

summary = pd.DataFrame({
    'anuncios': [len(df_diario), len(df_mensual)],
    'activos (activo = 1)': [df_diario['activo'].sum(), df_mensual['activo'].sum()],
    'inactivos (activo = 0)': [(df_diario['activo'] == 0).sum(), (df_mensual['activo'] == 0).sum()],
}, index=['df_diario', 'df_mensual'])
summary['% activos'] = (summary['activos (activo = 1)'] / summary['anuncios'] * 100).round(1)
print(f'df_diario:  {df_diario.shape}')
print(f'df_mensual: {df_mensual.shape}')
print(f"Anuncios para entrenar (df_diario con activo = 1): {df['in_training'].sum():,}")
summary

**Comparación: activos vs. inactivos dentro de cada DataFrame** (medianas, salvo los porcentajes).

In [ ]:
def describe_group(g):
    base = g['base_nightly_rate'].dropna()
    return pd.Series({
        'Anuncios': len(g),
        'Precio por noche (price)': g['price_num'].median(),
        'Tarifa base sin descuento (solo diario)': base.median() if len(base) else np.nan,
        'Noches mínimas': g['minimum_nights'].median(),
        '% casa o apto completo': (g['room_type'] == 'Entire home/apt').mean() * 100,
        'Capacidad': g['accommodates'].median(),
        'Habitaciones': g['bedrooms'].median(),
        'Baños': g['bathrooms'].median(),
        'Servicios': g['amenities_count'].median(),
        'Reseñas totales': g['number_of_reviews'].median(),
        '% sin ninguna reseña': (g['number_of_reviews'] == 0).mean() * 100,
        'Reseñas últimos 12 meses': g['number_of_reviews_ltm'].median(),
        'Ocupación estimada (días/año)': g['estimated_occupancy_l365d'].median(),
        'Días disponibles en el año': g['availability_365'].median(),
    })

groups = {
    'Diario · activo': df_diario[df_diario['activo'] == 1],
    'Diario · inactivo': df_diario[df_diario['activo'] == 0],
    'Mensual · activo': df_mensual[df_mensual['activo'] == 1],
    'Mensual · inactivo': df_mensual[df_mensual['activo'] == 0],
}
pd.DataFrame({name: describe_group(g) for name, g in groups.items()}).round(1)

In [ ]:
import matplotlib.pyplot as plt

names = list(groups)
labels = [n.replace(' · ', '\n') for n in names]
colors = ['#2a7ab9', '#d1495b', '#7fb3dd', '#eb9aa5']
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

box = axes[0].boxplot([groups[n]['price_num'] for n in names], showfliers=False, patch_artist=True)
for patch, color in zip(box['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_xticks(range(1, len(names) + 1), labels)
axes[0].set_yscale('log')
axes[0].set_title('Precio por noche (USD, escala log)')

axes[1].bar(labels, [groups[n]['amenities_count'].median() for n in names], color=colors)
axes[1].set_title('Servicios por anuncio (mediana)')

axes[2].bar(labels, [len(groups[n]) for n in names], color=colors)
axes[2].set_title('Número de anuncios')

plt.tight_layout()
plt.show()

### 3.8 · Quitar columnas repetidas o constantes
En `df_diario` y `df_mensual` se eliminan:
- **Repetidas:** `is_inactive` (es lo contrario de `activo`) y `quote_is_monthly_priced` (igual a `is_monthly`).
- **Constantes por el filtro:** las marcas que se usaron para crear `df_limpio` y para dividirlo valen lo mismo en todas las filas (`is_small_host`, `has_price`, `is_hotel`, `price_outlier`, `is_monthly`, `in_target_market`).
- **Vacía en `df_mensual`:** `base_nightly_rate`, que solo existe con precio diario.

Al final se verifica que no quede ninguna columna constante.

In [ ]:
REDUNDANT = ['is_inactive', 'quote_is_monthly_priced']
FILTER_FLAGS = ['is_small_host', 'has_price', 'is_hotel', 'price_outlier', 'is_monthly', 'in_target_market']

for name in ['df_diario', 'df_mensual']:
    data = globals()[name]
    drop = REDUNDANT + FILTER_FLAGS
    if name == 'df_mensual':
        drop = drop + ['base_nightly_rate']
    constant = [c for c in FILTER_FLAGS if data[c].nunique(dropna=False) == 1]
    assert constant == FILTER_FLAGS, f'{name}: estas marcas no son constantes: {set(FILTER_FLAGS) - set(constant)}'
    data = data.drop(columns=drop)

    still_constant = [c for c in data.columns if data[c].nunique(dropna=False) <= 1]
    assert not still_constant, f'{name}: quedan columnas constantes: {still_constant}'
    globals()[name] = data
    print(f'{name}: {data.shape}  (se quitaron {len(drop)} columnas)')

## 4 · Análisis de variables y del precio
Esta sección reorganiza el análisis de Daniel Babativa (rama `daniel-babativa`) en este orden:

1. **Correlación** de cada variable con el **precio original** (`price_num`).
2. **Limpieza de variables:** de cada pareja muy correlacionada se deja solo la que mejor explica el precio.
3. **ANOVA** del precio original entre los 4 grupos (diario/mensual × activo/inactivo): qué grupos son distintos y cuáles se pueden unir.
4. **Distribución** del precio de cada grupo resultante (precio original vs. logaritmo), con gráficas y pruebas estadísticas.
5. **DataFrames finales** para el modelo.

Se trabaja sobre `df_limpio` (16.158 anuncios: hosts con 1–2 anuncios, con precio, sin hoteles ni precios extremos).

### 4.1 · Correlación de cada variable con el precio original
**Variables candidatas a predictor.** Antes de medir correlaciones se dejan fuera, por definición, las columnas que no pueden ser predictores:

| Columna | Por qué no es predictor |
|---|---|
| `id` | Identificador, no describe la propiedad. |
| `base_nightly_rate` | Es otra medida del precio (variable objetivo alternativa): usarla filtraría la respuesta. |
| Actividad: `number_of_reviews`, `number_of_reviews_ltm`, `last_review`, `availability_365`, `estimated_occupancy_l365d`, `has_availability` | Resultados de operar el anuncio; un anfitrión nuevo no los conoce. |
| `bedrooms_original`, `beds_original`, `*_imputed`, `bedrooms_gt_guests` | Control de calidad de la limpieza. |
| `bathrooms_text` | Texto: ya está convertido en `bathrooms` y `bathroom_shared`. |
| `neighbourhood_cleansed`, `property_type` | Versiones sin agrupar: se usan `neighbourhood_model` y `property_type_model`. |
| `calculated_host_listings_count`, marcas de limpieza, `activo` | Filtros de la población, no características de la propiedad. |

**Medidas de correlación:**
- Variables numéricas y servicios (0/1): **Pearson** y **Spearman**. Como el precio es muy asimétrico (sección 3.7), el criterio para decidir es **Spearman**, que usa rangos y no se deja arrastrar por los precios extremos.
- Variables categóricas: **razón de correlación η (eta)**, que mide qué parte de la variación del precio explican las categorías (η² = proporción explicada).

In [ ]:
from scipy import stats

am_cols = [c for c in df_limpio.columns if c.startswith('am_')]
NUMERIC_CANDIDATES = ['accommodates', 'bedrooms', 'beds', 'bathrooms', 'bathroom_shared', 'amenities_count',
                      'minimum_nights', 'maximum_nights', 'quote_nights', 'latitude', 'longitude'] + am_cols
CATEGORICAL_CANDIDATES = ['neighbourhood_model', 'neighbourhood_group_cleansed', 'property_type_model', 'room_type']

price = df_limpio['price_num']

def correlation_ratio(categories, values):
    # η: raíz de la proporción de la varianza de `values` explicada por las categorías
    groups = values.groupby(categories)
    between = (groups.count() * (groups.mean() - values.mean()) ** 2).sum()
    total = ((values - values.mean()) ** 2).sum()
    return np.sqrt(between / total)

corr_table = pd.DataFrame({
    'tipo': 'numérica',
    'Pearson': df_limpio[NUMERIC_CANDIDATES].corrwith(price),
    'Spearman': df_limpio[NUMERIC_CANDIDATES].corrwith(price, method='spearman'),
})
cat_table = pd.DataFrame({
    'tipo': 'categórica',
    'eta (η)': [correlation_ratio(df_limpio[c], price) for c in CATEGORICAL_CANDIDATES],
}, index=CATEGORICAL_CANDIDATES)
corr_table = pd.concat([corr_table, cat_table])
corr_table['fuerza (|Spearman| o η)'] = corr_table['Spearman'].abs().fillna(corr_table['eta (η)'])
corr_table = corr_table.sort_values('fuerza (|Spearman| o η)', ascending=False)
corr_table.round(3)

In [ ]:
import matplotlib.pyplot as plt

# Correlación de Spearman entre las variables numéricas (mitad inferior); se rotulan las parejas con |ρ| ≥ 0,7
spearman_matrix = df_limpio[NUMERIC_CANDIDATES].corr(method='spearman')
values = spearman_matrix.to_numpy()
masked = np.ma.masked_where(np.triu(np.ones_like(values, dtype=bool), k=1) | ~np.isfinite(values), values)
cmap = plt.get_cmap('coolwarm').copy()
cmap.set_bad('white')

n_vars = len(NUMERIC_CANDIDATES)
fig, ax = plt.subplots(figsize=(15, 13))
image = ax.imshow(masked, cmap=cmap, vmin=-1, vmax=1)
ax.set_xticks(range(n_vars), NUMERIC_CANDIDATES, rotation=90, fontsize=8)
ax.set_yticks(range(n_vars), NUMERIC_CANDIDATES, fontsize=8)
for i in range(n_vars):
    for j in range(i):
        if abs(values[i, j]) >= 0.7:
            ax.text(j, i, f'{values[i, j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(image, ax=ax, label='Correlación de Spearman')
ax.set_title('Correlación de Spearman entre las variables numéricas candidatas')
plt.tight_layout()
plt.show()

### 4.2 · Dejar solo la variable que mejor explica el precio
**Regla:** si dos variables tienen una correlación de **|ρ| ≥ 0,8** entre ellas, se conserva la que tiene **mayor |Spearman| con el precio original** y se elimina la otra. Las parejas se revisan de la más correlacionada a la menos.

**Excepción:** `accommodates` (capacidad) y `bedrooms` (habitaciones) son características obligatorias del Project Charter y entradas del tablero, así que no se eliminan aunque estén correlacionadas entre sí; la pareja se reporta.

**Categóricas anidadas:** si dos variables categóricas tienen una asociación de **V de Cramér ≥ 0,8** (una casi determina a la otra), se conserva la de mayor η con el precio.

In [ ]:
CORR_THRESHOLD = 0.8
PROTECTED = {'accommodates', 'bedrooms'}            # características obligatorias del Project Charter
strength = corr_table['fuerza (|Spearman| o η)']

# Parejas numéricas muy correlacionadas, de mayor a menor
pairs = []
for i, a in enumerate(NUMERIC_CANDIDATES):
    for b in NUMERIC_CANDIDATES[i + 1:]:
        rho = spearman_matrix.loc[a, b]
        if abs(rho) >= CORR_THRESHOLD:
            pairs.append((a, b, rho))

# Parejas categóricas: V de Cramér
def cramers_v(x, y):
    table = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(table, correction=False)[0]
    return np.sqrt(chi2 / (table.to_numpy().sum() * (min(table.shape) - 1)))

for i, a in enumerate(CATEGORICAL_CANDIDATES):
    for b in CATEGORICAL_CANDIDATES[i + 1:]:
        v = cramers_v(df_limpio[a], df_limpio[b])
        if v >= CORR_THRESHOLD:
            pairs.append((a, b, v))

dropped, decisions = set(), []
for a, b, assoc in sorted(pairs, key=lambda t: -abs(t[2])):
    if a in dropped or b in dropped:
        decisions.append((a, b, assoc, '—', 'Ya se eliminó una de las dos'))
        continue
    if {a, b} <= PROTECTED:
        decisions.append((a, b, assoc, '—', 'Se conservan ambas (Project Charter)'))
        continue
    keep, drop = (a, b) if strength[a] >= strength[b] else (b, a)
    if drop in PROTECTED:
        keep, drop = drop, keep
    dropped.add(drop)
    decisions.append((a, b, assoc, drop, f'Se conserva {keep} ({strength[keep]:.3f} vs {strength[drop]:.3f})'))

decision_table = pd.DataFrame(decisions, columns=['variable 1', 'variable 2', 'asociación', 'se elimina', 'motivo'])
display(decision_table.round(3))

SELECTED_NUMERIC = [c for c in NUMERIC_CANDIDATES if c not in dropped]
SELECTED_CATEGORICAL = [c for c in CATEGORICAL_CANDIDATES if c not in dropped]
SELECTED_FEATURES = SELECTED_NUMERIC + SELECTED_CATEGORICAL
print(f'Variables eliminadas ({len(dropped)}): {sorted(dropped)}')
print(f'Variables seleccionadas ({len(SELECTED_FEATURES)}): {len(SELECTED_NUMERIC)} numéricas y {len(SELECTED_CATEGORICAL)} categóricas')
print(SELECTED_FEATURES)

### 4.3 · ANOVA: ¿los precios de los 4 grupos son diferentes?
Se compara el **precio original** entre los 4 grupos de la sección 3.7:

| Prueba | Pregunta |
|---|---|
| **ANOVA de una vía** | ¿Las medias de los 4 grupos son iguales? |
| **Levene** | ¿Las varianzas son iguales? (supuesto del ANOVA clásico) |
| **ANOVA de Welch** | Igual que el ANOVA, pero sin suponer varianzas iguales |
| **Kruskal-Wallis** | Versión no paramétrica (no supone normalidad) |
| **Tukey HSD** y **t de Welch con corrección de Bonferroni** | ¿Qué parejas de grupos son distintas? |
| **d de Cohen** | ¿Qué tan grande es la diferencia? (0,2 pequeña, 0,5 mediana, 0,8 grande) |

**Regla para unir grupos:** con miles de datos, casi cualquier diferencia sale estadísticamente significativa, aunque sea mínima. Por eso dos grupos se **unen** si la diferencia **no es significativa** (Bonferroni, α = 0,05) **o** si su tamaño es **despreciable** (|d de Cohen| < 0,2). Si es significativa y de tamaño al menos pequeño, se **separan**.

In [ ]:
df_limpio['grupo'] = (np.where(df_limpio['is_monthly'] == 1, 'Mensual', 'Diario') + ' · '
                      + np.where(df_limpio['activo'] == 1, 'activo', 'inactivo'))
GROUP_ORDER = ['Diario · activo', 'Diario · inactivo', 'Mensual · activo', 'Mensual · inactivo']
samples = {g: df_limpio.loc[df_limpio['grupo'] == g, 'price_num'].to_numpy() for g in GROUP_ORDER}

summary = pd.DataFrame({g: {'anuncios': len(s), 'media': s.mean(), 'mediana': np.median(s),
                            'desviación': s.std(ddof=1)} for g, s in samples.items()}).T
display(summary.round(1))

def welch_anova(groups):
    n = np.array([len(g) for g in groups]); m = np.array([g.mean() for g in groups])
    w = n / np.array([g.var(ddof=1) for g in groups]); k = len(groups)
    mean_w = (w * m).sum() / w.sum()
    a = (w * (m - mean_w) ** 2).sum() / (k - 1)
    tmp = ((1 - w / w.sum()) ** 2 / (n - 1)).sum()
    f = a / (1 + 2 * (k - 2) / (k ** 2 - 1) * tmp)
    df2 = (k ** 2 - 1) / (3 * tmp)
    return f, k - 1, df2, stats.f.sf(f, k - 1, df2)

groups = list(samples.values())
anova = stats.f_oneway(*groups)
levene = stats.levene(*groups)
f_w, df1_w, df2_w, p_w = welch_anova(groups)
kruskal = stats.kruskal(*groups)
global_tests = pd.DataFrame({
    'estadístico': [anova.statistic, levene.statistic, f_w, kruskal.statistic],
    'valor p': [anova.pvalue, levene.pvalue, p_w, kruskal.pvalue],
}, index=['ANOVA de una vía (F)', 'Levene (varianzas iguales)', 'ANOVA de Welch (F)', 'Kruskal-Wallis (H)'])
global_tests['conclusión (α = 0,05)'] = np.where(global_tests['valor p'] < 0.05, 'Se rechaza H0', 'No se rechaza H0')
global_tests

In [ ]:
from itertools import combinations

tukey = stats.tukey_hsd(*groups)
n_pairs = len(GROUP_ORDER) * (len(GROUP_ORDER) - 1) // 2
rows = []
for (i, a), (j, b) in combinations(enumerate(GROUP_ORDER), 2):
    x, y = samples[a], samples[b]
    pooled_sd = np.sqrt(((len(x) - 1) * x.var(ddof=1) + (len(y) - 1) * y.var(ddof=1)) / (len(x) + len(y) - 2))
    d = (x.mean() - y.mean()) / pooled_sd
    p_welch = min(stats.ttest_ind(x, y, equal_var=False).pvalue * n_pairs, 1.0)    # Bonferroni
    significant = p_welch < 0.05
    merge = (not significant) or abs(d) < 0.2
    rows.append({'grupo 1': a, 'grupo 2': b, 'diferencia de medias': x.mean() - y.mean(),
                 'p Tukey': tukey.pvalue[i, j], 'p Welch (Bonferroni)': p_welch, 'd de Cohen': d,
                 'decisión': 'Unir' if merge else 'Separar'})
pairwise = pd.DataFrame(rows)
display(pairwise.round(4))

# Unir los grupos conectados por una decisión "Unir"
parent = {g: g for g in GROUP_ORDER}
def find(g):
    while parent[g] != g:
        g = parent[g]
    return g
for _, r in pairwise[pairwise['decisión'] == 'Unir'].iterrows():
    parent[find(r['grupo 2'])] = find(r['grupo 1'])

clusters = {}
for g in GROUP_ORDER:
    clusters.setdefault(find(g), []).append(g)
def cluster_name(members):
    if len(members) == 1:
        return members[0]
    kinds = {m.split(' · ')[0] for m in members}
    return kinds.pop() if len(kinds) == 1 else ' + '.join(members)
GROUP_MAP = {m: cluster_name(ms) for ms in clusters.values() for m in ms}
df_limpio['grupo_precio'] = df_limpio['grupo'].map(GROUP_MAP)

print('Grupos de precio resultantes:')
print(df_limpio['grupo_precio'].value_counts().reindex(dict.fromkeys(GROUP_MAP.values())).to_string())

### 4.4 · ¿A qué distribución se ajusta el precio de cada grupo?
Para cada grupo resultante se compara el **precio original** y su **logaritmo** contra una distribución normal:

- **Asimetría:** 0 = simétrica; mientras más lejos de 0, más cola hacia un lado.
- **Kolmogorov-Smirnov (KS):** distancia máxima *D* entre la distribución de los datos y la normal ajustada (menor = más parecida), con su valor p.
- **Shapiro-Wilk:** estadístico *W* (1 = normal perfecta), sobre una muestra de hasta 5.000 datos.
- **Gráficas:** histograma con la normal ajustada, Q-Q (cuantiles) y P-P (probabilidades).

**Recomendación:** se usa el **logaritmo** si acerca la asimetría a 0 **y** reduce la distancia KS; si no, se deja el precio original.

> Con miles de datos, KS y Shapiro-Wilk rechazan la normalidad exacta casi siempre (valor p ≈ 0) aunque la aproximación sea muy buena. Por eso la decisión se basa en **D, W, la asimetría y las gráficas**, no solo en el valor p. Además, como la media y la desviación se estiman de los mismos datos, el valor p de KS es solo aproximado.

In [ ]:
rng = np.random.default_rng(42)

def normality(values):
    mu, sd = values.mean(), values.std(ddof=1)
    ks = stats.kstest(values, stats.norm(mu, sd).cdf)
    sample = rng.choice(values, size=min(len(values), 5000), replace=False)
    sw = stats.shapiro(sample)
    return {'asimetría': stats.skew(values), 'KS D': ks.statistic, 'KS valor p': ks.pvalue,
            'Shapiro W': sw.statistic, 'Shapiro valor p': sw.pvalue}

PRICE_GROUPS = list(dict.fromkeys(GROUP_MAP.values()))
dist_rows, recommendations = [], {}
for g in PRICE_GROUPS:
    p = df_limpio.loc[df_limpio['grupo_precio'] == g, 'price_num'].to_numpy()
    original, logged = normality(p), normality(np.log(p))
    use_log = abs(logged['asimetría']) < abs(original['asimetría']) and logged['KS D'] < original['KS D']
    recommendations[g] = 'logaritmo' if use_log else 'original'
    dist_rows.append({'grupo': g, 'escala': 'precio original', **original})
    dist_rows.append({'grupo': g, 'escala': 'log(precio)', **logged})

distribution_table = pd.DataFrame(dist_rows).set_index(['grupo', 'escala'])
display(distribution_table.round(4))
print('Recomendación por grupo:')
for g, rec in recommendations.items():
    print(f'  {g}: usar el precio {"en logaritmo" if rec == "logaritmo" else "original"}')

In [ ]:
def plot_normality(ax_row, values, title):
    values = np.sort(values)
    mu, sd = values.mean(), values.std(ddof=1)
    probs = (np.arange(1, len(values) + 1) - 0.5) / len(values)
    ax_row[0].hist(values, bins=50, density=True, alpha=0.65)
    x = np.linspace(values.min(), values.max(), 300)
    ax_row[0].plot(x, stats.norm.pdf(x, mu, sd), 'r')
    ax_row[0].set_title(f'Histograma: {title}', fontsize=10)
    ax_row[1].scatter(stats.norm.ppf(probs, mu, sd), values, s=4, alpha=0.5)
    lims = [min(values.min(), mu - 4 * sd), max(values.max(), mu + 4 * sd)]
    ax_row[1].plot(lims, lims, 'r--')
    ax_row[1].set_title(f'Q-Q: {title}', fontsize=10)
    ax_row[2].scatter(stats.norm.cdf(values, mu, sd), probs, s=4, alpha=0.5)
    ax_row[2].plot([0, 1], [0, 1], 'r--')
    ax_row[2].set_title(f'P-P: {title}', fontsize=10)

fig, axes = plt.subplots(2 * len(PRICE_GROUPS), 3, figsize=(16, 3.6 * 2 * len(PRICE_GROUPS)))
for k, g in enumerate(PRICE_GROUPS):
    p = df_limpio.loc[df_limpio['grupo_precio'] == g, 'price_num'].to_numpy()
    plot_normality(axes[2 * k], p, f'{g} · precio')
    plot_normality(axes[2 * k + 1], np.log(p), f'{g} · log(precio)')
plt.tight_layout()
plt.show()

### 4.5 · DataFrames finales para el modelo
Se crea un DataFrame por **grupo de precio** resultante del ANOVA. Cada uno contiene:
- `id` y `grupo_precio`,
- `activo` (se conserva para poder filtrar, aunque el grupo junte activos e inactivos),
- la variable objetivo: `price_num` y, si la sección 4.4 lo recomienda, `price_log`,
- las **variables seleccionadas** en la sección 4.2.

In [ ]:
# 10 anuncios no traen noches mínimas ni máximas. Mínimas = noches cotizadas (coinciden en el 93,5 % de los
# anuncios, sección 2); máximas = mediana del grupo. Se marca con nights_imputed.
df_limpio['nights_imputed'] = df_limpio['minimum_nights'].isna().astype(int)
df_limpio['minimum_nights'] = df_limpio['minimum_nights'].fillna(df_limpio['quote_nights'])
df_limpio['maximum_nights'] = df_limpio['maximum_nights'].fillna(
    df_limpio.groupby('grupo_precio')['maximum_nights'].transform('median'))
print(f"Noches mínimas/máximas rellenadas: {df_limpio['nights_imputed'].sum()} anuncios")

model_dfs = {}
for g in PRICE_GROUPS:
    data = df_limpio.loc[df_limpio['grupo_precio'] == g, ['id', 'grupo_precio', 'activo', 'price_num'] + SELECTED_FEATURES].copy()
    if recommendations[g] == 'logaritmo':
        data.insert(data.columns.get_loc('price_num') + 1, 'price_log', np.log(data['price_num']))
    constant = [c for c in SELECTED_FEATURES if data[c].nunique(dropna=False) <= 1]
    data = data.drop(columns=constant)                    # p. ej. variables que no varían dentro del grupo
    assert data[[c for c in data.columns if c in SELECTED_FEATURES]].isna().sum().sum() == 0, f'{g}: hay vacíos'
    model_dfs[g] = data.reset_index(drop=True)

NAMES = {g: 'df_modelo_' + g.lower().replace(' · ', '_').replace(' + ', '_y_').replace(' ', '_') for g in PRICE_GROUPS}
for g, name in NAMES.items():
    globals()[name] = model_dfs[g]
NAMES

In [ ]:
# Shape y contenido de cada DataFrame resultante
for g, name in NAMES.items():
    data = model_dfs[g]
    target = 'price_log (logaritmo del precio)' if 'price_log' in data else 'price_num (precio original)'
    activos = int(data['activo'].sum())
    print(f'{name}: {data.shape}')
    print(f'  Contiene: anuncios del grupo "{g}" ({len(data):,} anuncios; {activos:,} activos y {len(data) - activos:,} inactivos).')
    print(f'  Variable objetivo recomendada: {target}.')
    print(f'  Predictores: {data.shape[1] - 4 - ("price_log" in data)} variables seleccionadas en 4.2.')
    print()